In [4]:
import os
import cv2
import numpy as np
import torch
import pandas as pd
from tqdm import tqdm
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Choose Model to Evaluate

## DLinkNet

In [54]:
import torch
import torch.nn as nn
from torchvision.models import resnet34, ResNet34_Weights

class DBlock(nn.Module):
    def __init__(self, channels):
        super(DBlock, self).__init__()
        self.dilate1 = nn.Conv2d(channels, channels, kernel_size=3, dilation=1, padding=1)
        self.dilate2 = nn.Conv2d(channels, channels, kernel_size=3, dilation=2, padding=2)
        self.dilate3 = nn.Conv2d(channels, channels, kernel_size=3, dilation=4, padding=4)
        self.dilate4 = nn.Conv2d(channels, channels, kernel_size=3, dilation=8, padding=8)
        
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        d1 = self.relu(self.dilate1(x))
        d2 = self.relu(self.dilate2(d1))
        d3 = self.relu(self.dilate3(d2))
        d4 = self.relu(self.dilate4(d3))
        
        out = x + d1 + d2 + d3 + d4
        return out


class DecoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DecoderBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, in_channels // 4, kernel_size=1)
        self.bn1 = nn.BatchNorm2d(in_channels // 4)  # norm1 -> bn1
        self.relu1 = nn.ReLU(inplace=True)

        self.deconv3d = nn.ConvTranspose3d(           # deconv2 -> deconv3d
            in_channels // 4, in_channels // 4, 
            kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1), output_padding=(0, 1, 1)
        )
        self.bn2 = nn.BatchNorm2d(in_channels // 4)   # norm2 -> bn2
        self.relu2 = nn.ReLU(inplace=True)

        self.conv3 = nn.Conv2d(in_channels // 4, out_channels, kernel_size=1)
        self.bn3 = nn.BatchNorm2d(out_channels)        # norm3 -> bn3
        self.relu3 = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        
        x = x.unsqueeze(2)
        x = self.deconv3d(x)
        x = x.squeeze(2)
        
        x = self.bn2(x)
        x = self.relu2(x)
        
        x = self.conv3(x)
        x = self.bn3(x)
        x = self.relu3(x)
        return x


class DLinkNet34(nn.Module):
    def __init__(self, num_classes=1, pretrained=True):
        super(DLinkNet34, self).__init__()
        
        weights = ResNet34_Weights.DEFAULT if pretrained else None
        base_resnet = resnet34(weights=weights)
        
        self.firstconv = base_resnet.conv1
        self.firstbn = base_resnet.bn1
        self.firstrelu = base_resnet.relu
        self.firstmaxpool = base_resnet.maxpool
        
        self.encoder1 = base_resnet.layer1  
        self.encoder2 = base_resnet.layer2  
        self.encoder3 = base_resnet.layer3  
        self.encoder4 = base_resnet.layer4  

        self.dblock = DBlock(512)

        self.decoder4 = DecoderBlock(512, 256)
        self.decoder3 = DecoderBlock(256, 128)
        self.decoder2 = DecoderBlock(128, 64)
        self.decoder1 = DecoderBlock(64, 64)

        self.finaldeconv1 = nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1)
        self.finalrelu1 = nn.ReLU(inplace=True)
        self.finalconv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.finalrelu2 = nn.ReLU(inplace=True)
        self.finalconv3 = nn.Conv2d(32, num_classes, kernel_size=3, padding=1)
        
    def forward(self, x):
        x = self.firstconv(x)
        x = self.firstbn(x)
        x = self.firstrelu(x)
        x = self.firstmaxpool(x)

        e1 = self.encoder1(x)  
        e2 = self.encoder2(e1) 
        e3 = self.encoder3(e2) 
        e4 = self.encoder4(e3) 

        center = self.dblock(e4)

        d4 = self.decoder4(center) + e3
        d3 = self.decoder3(d4) + e2
        d2 = self.decoder2(d3) + e1
        d1 = self.decoder1(d2)

        out = self.finaldeconv1(d1)
        out = self.finalrelu1(out)
        out = self.finalconv2(out)
        out = self.finalrelu2(out)
        out = self.finalconv3(out)

        return out

In [55]:
model_path = '/kaggle/input/models/vuongtran11233/dlinknet-test/pytorch/default/1/best_dlinknet.pth'
model = DLinkNet34(num_classes=1)
# 1. Load the dictionary file
checkpoint = torch.load(model_path, map_location=device)

# Handle cases where you saved the entire checkpoint dict or just the state_dict
state_dict = checkpoint['model_state_dict']

# 2. Create a clean dictionary without the "module." prefix
from collections import OrderedDict
clean_state_dict = OrderedDict()

for key, value in state_dict.items():
    # If it starts with module., strip it out
    if key.startswith('module.'):
        clean_name = key[7:]  # 'module.' is 7 characters long
        clean_state_dict[clean_name] = value
    else:
        clean_state_dict[key] = value

# 3. Load the clean state dict into your model
model.load_state_dict(clean_state_dict)

#model = DLinkNet34(num_classes=1)
#checkpoint = torch.load(model_path, map_location=device)
#model.load_state_dict(checkpoint['model_state_dict'])

model.to(device)
model.eval()

DLinkNet34(
  (firstconv): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (firstbn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (firstrelu): ReLU(inplace=True)
  (firstmaxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (encoder1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu

## DeepLabV3+

In [36]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet34, ResNet34_Weights

class ASPPConv(nn.Sequential):
    """Standard Conv block with dilation for ASPP"""
    def __init__(self, in_channels, out_channels, dilation):
        super(ASPPConv, self).__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=dilation, dilation=dilation, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

class ASPPPooling(nn.Sequential):
    """Global Average Pooling branch for ASPP"""
    def __init__(self, in_channels, out_channels):
        super(ASPPPooling, self).__init__(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        size = x.shape[-2:]
        out = super(ASPPPooling, self).forward(x)
        return F.interpolate(out, size=size, mode='bilinear', align_corners=False)

class ASPP(nn.Module):
    """Atrous Spatial Pyramid Pooling Module adapted for ResNet-34 (512 input channels)"""
    def __init__(self, in_channels=512, out_channels=256, rates=[6, 12, 18]):
        super(ASPP, self).__init__()
        modules = []
        # 1. 1x1 Convolution branch
        modules.append(nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        ))
        
        # 2. Three Dilated 3x3 Convolution branches
        for rate in rates:
            modules.append(ASPPConv(in_channels, out_channels, rate))
            
        # 3. Image Pooling branch
        modules.append(ASPPPooling(in_channels, out_channels))
        
        self.convs = nn.ModuleList(modules)
        
        # 4. Final Projection layer fusing all 5 branches
        self.project = nn.Sequential(
            nn.Conv2d(len(modules) * out_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5)
        )

    def forward(self, x):
        res = []
        for conv in self.convs:
            res.append(conv(x))
        res = torch.cat(res, dim=1)
        return self.project(res)


class DeepLabV3PlusResNet34(nn.Module):
    def __init__(self, num_classes=1, pretrained=True):
        super(DeepLabV3PlusResNet34, self).__init__()
        
        # 1. ENCODER: Extract backbone from ResNet-34
        weights = ResNet34_Weights.DEFAULT if pretrained else None
        base_resnet = resnet34(weights=weights)
        
        self.initial_blocks = nn.Sequential(
            base_resnet.conv1,
            base_resnet.bn1,
            base_resnet.relu
        )
        self.maxpool = base_resnet.maxpool
        
        self.layer1 = base_resnet.layer1  # Low-level features: 64 channels, 1/4 resolution
        self.layer2 = base_resnet.layer2  # 128 channels, 1/8 resolution
        self.layer3 = base_resnet.layer3  # 256 channels, 1/16 resolution
        self.layer4 = base_resnet.layer4  # High-level features: 512 channels, 1/32 resolution

        # 2. BOTTLENECK: Multi-scale context gathering via ASPP
        self.aspp = ASPP(in_channels=512, out_channels=256)

        # 3. DECODER: Feature refinement and fusion
        # Low-level projection (reduces channels from early encoder to avoid overpowering high-level features)
        self.low_level_project = nn.Sequential(
            nn.Conv2d(64, 48, kernel_size=1, bias=False),
            nn.BatchNorm2d(48),
            nn.ReLU(inplace=True)
        )
        
        # Main Decoder Refining Layers (processes fused 256 + 48 = 304 channels)
        self.decoder_head = nn.Sequential(
            nn.Conv2d(304, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Conv2d(256, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Conv2d(256, num_classes, kernel_size=1)
        )

    def forward(self, x):
        input_size = x.shape[-2:]  # Keep original H, W for final upsampling
        
        # --- Encoder Forward ---
        x = self.initial_blocks(x)
        low_level_features = self.layer1(x)  # Retain for decoder connection (1/4 size)
        
        x = self.maxpool(low_level_features)
        x = self.layer2(x)
        x = self.layer3(x)
        high_level_features = self.layer4(x) # 512 channels (1/32 size)

        # --- ASPP Bottleneck ---
        aspp_features = self.aspp(high_level_features) # 256 channels
        
        # --- Decoder Forward ---
        # 1. Upsample high-level ASPP features 8x to meet low-level size (1/4 size)
        aspp_features_upsampled = F.interpolate(
            aspp_features, size=low_level_features.shape[-2:], 
            mode='bilinear', align_corners=False
        )
        
        # 2. Process low-level features
        low_features_projected = self.low_level_project(low_level_features)
        
        # 3. Concatenate along channel dimension (U-Net style channel stacking)
        fused_features = torch.cat([aspp_features_upsampled, low_features_projected], dim=1)
        
        # 4. Refine via convolution blocks
        decoder_output = self.decoder_head(fused_features)
        
        # --- Final Upsampling to Match Input Resolution ---
        output = F.interpolate(decoder_output, size=input_size, mode='bilinear', align_corners=False)
        return output

In [37]:
model_path = '/kaggle/input/models/vuongtran11233/test-deeplabv3-/pytorch/default/1/best_deeplabv3.pth'
# === Load model ===
model = DeepLabV3PlusResNet34(num_classes=1)
# 1. Load the dictionary file
checkpoint = torch.load(model_path, map_location=device)

# Handle cases where you saved the entire checkpoint dict or just the state_dict
state_dict = checkpoint['model_state_dict']

# 2. Create a clean dictionary without the "module." prefix
from collections import OrderedDict
clean_state_dict = OrderedDict()

for key, value in state_dict.items():
    # If it starts with module., strip it out
    if key.startswith('module.'):
        clean_name = key[7:]  # 'module.' is 7 characters long
        clean_state_dict[clean_name] = value
    else:
        clean_state_dict[key] = value

# 3. Load the clean state dict into your model
model.load_state_dict(clean_state_dict)

#model = DLinkNet34(num_classes=1)
#checkpoint = torch.load(model_path, map_location=device)
#model.load_state_dict(checkpoint['model_state_dict'])

model.to(device)
model.eval()

DeepLabV3PlusResNet34(
  (initial_blocks): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running

## SegFormer

In [22]:
model_path = '/kaggle/input/models/vuongtran11233/test-segformer/pytorch/default/1/best_segformer_b2 (1).pth'
from collections import OrderedDict
import torch
from collections import OrderedDict
from transformers import SegformerForSemanticSegmentation

# 1. Initialize the base architecture from Hugging Face
# This sets up the structure and handles the new class count (num_labels=1)
model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/mit-b2",
    num_labels=1,
    ignore_mismatched_sizes=True
)

# 2. Load your Kaggle .pth checkpoint
input_model_path = model_path
checkpoint = torch.load(input_model_path, map_location=device)

# 3. Handle the dictionary check (depending on how you saved it)
# If you didn't save it with a 'model_state_dict' wrapper, just use checkpoint directly
if 'model_state_dict' in checkpoint:
    state_dict = checkpoint['model_state_dict']
else:
    state_dict = checkpoint

# 4. Strip the 'module.' prefix from DataParallel
clean_state_dict = OrderedDict()
for key, value in state_dict.items():
    if key.startswith('module.'):
        clean_state_dict[key[7:]] = value
    else:
        clean_state_dict[key] = value

# 5. Load the weights into the model
# CRITICAL: strict=False allows it to ignore mismatched head weights smoothly
missing_keys, unexpected_keys = model.load_state_dict(clean_state_dict, strict=False)

# Optional: Print to ensure everything loaded smoothly except the classification head
print("Missing keys:", missing_keys)
print("Unexpected keys:", unexpected_keys)

model.to(device)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/99.0M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/364 [00:00<?, ?it/s]

SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b2
Key                                           | Status     | 
----------------------------------------------+------------+-
classifier.bias                               | UNEXPECTED | 
classifier.weight                             | UNEXPECTED | 
decode_head.linear_c.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.batch_norm.num_batches_tracked    | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.batch_norm.running_var            | MISSING    | 
decode_head.batch_norm.weight                 | MISSING    | 
decode_head.classifier.weight                 | MISSING    | 
decode_head.linear_fuse.weight                | MISSING    | 
decode_head.batch_norm.running_mean           | MISSING    | 
decode_head.batch_norm.bias                   | MISSING    | 
decode_head.classifier.bias                   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different ta

model.safetensors:   0%|          | 0.00/98.9M [00:00<?, ?B/s]

Missing keys: []
Unexpected keys: []


SegformerForSemanticSegmentation(
  (segformer): SegformerModel(
    (encoder): SegformerEncoder(
      (patch_embeddings): ModuleList(
        (0): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(3, 64, kernel_size=(7, 7), stride=(4, 4), padding=(3, 3))
          (layer_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        )
        (1): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        )
        (2): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(128, 320, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
        )
        (3): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(320, 512, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)

# Evaluation

In [57]:
csv_path = '/kaggle/input/datasets/vuongtran11233/new-division/data_division.csv'
image_root = '/kaggle/input/datasets/vuongtran11233/combined-dataset/combined_dataset/images'
mask_root = '/kaggle/input/datasets/vuongtran11233/combined-dataset/combined_dataset/masks'
threshold = 0.5

In [58]:
# === Load file ảnh
df = pd.read_csv(csv_path)
test_df = df[df['split'] == 'test']

# Lấy danh sách file ảnh và mask
image_filenames = test_df['filename'].tolist()
mask_filenames = test_df['maskname'].tolist()

In [59]:
# Model selection prompt
print("Available models: 'dlinknet', 'segformer', 'deeplabv3'")
selected_model = input("Please enter the model you want to use: ").strip().lower()

# Validate the user's choice
valid_models = ['dlinknet', 'segformer', 'deeplabv3']

while selected_model not in valid_models:
    print(f"\n[Error] Invalid choice or no selection made.")
    print(f"You must choose exactly one among: {', '.join(valid_models)}")
    selected_model = input("Please enter a valid model name: ").strip().lower()

print(f"\n Success: '{selected_model}' selected. Proceeding with configuration...")

Available models: 'dlinknet', 'segformer', 'deeplabv3'


Please enter the model you want to use:  dlinknet



 Success: 'dlinknet' selected. Proceeding with configuration...


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

# ==========================================
# 1. DEFINE A FAST MULTI-THREADED DATASET
# ==========================================
class RoadEvaluationDataset(Dataset):
    def __init__(self, img_filenames, mask_filenames, img_root, mask_root):
        self.img_filenames = img_filenames
        self.mask_filenames = mask_filenames
        self.img_root = img_root
        self.mask_root = mask_root

    def __len__(self):
        return len(self.img_filenames)

    def __getitem__(self, idx):
        # Generate full paths
        img_path = os.path.join(self.img_root, self.img_filenames[idx])
        mask_path = os.path.join(self.mask_root, self.mask_filenames[idx])

        # Read Image
        image = cv2.imread(img_path)
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if selected_model != 'dlinknet':
            image_rgb = cv2.resize(image_rgb, (512, 512), interpolation=cv2.INTER_LINEAR)
        
        # Normalize and transform Image (matching training)
        input_tensor = image_rgb.astype(np.float32) / 255.0
        input_tensor = input_tensor * 3.2 - 1.6
        input_tensor = np.transpose(input_tensor, (2, 0, 1)) # (C, H, W)

        # Read and process Mask (matching training normalization perfectly)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if selected_model != 'dlinknet':
            mask = cv2.resize(mask, (512, 512), interpolation=cv2.INTER_NEAREST)
        mask_normalized = mask.astype(np.float32) / 255.0
        mask_binary = (mask_normalized > 0.5).astype(np.float32)

        return torch.tensor(input_tensor), torch.tensor(mask_binary).unsqueeze(0)


# ==========================================
# 2. INITIALIZE DATALOADER & CONTAINERS
# ==========================================
eval_dataset = RoadEvaluationDataset(image_filenames, mask_filenames, image_root, mask_root)

# Using batch_size=16 or 32 utilizes the T4 GPU efficiently.
# num_workers=2 enables asynchronous background image loading on CPU threads.
eval_loader = DataLoader(
    eval_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [64]:
import numpy as np
import torch
import torch.nn.functional as F

def get_batch_stats(pred_mask, gt_mask):
    """Returns TP, TN, FP, FN, intersection, union, and total pixels for binary masks."""
    pred_flat = pred_mask.view(-1).long()
    gt_flat = gt_mask.view(-1).long()

    tp = torch.sum((pred_flat == 1) & (gt_flat == 1)).item()
    tn = torch.sum((pred_flat == 0) & (gt_flat == 0)).item()
    fp = torch.sum((pred_flat == 1) & (gt_flat == 0)).item()
    fn = torch.sum((pred_flat == 0) & (gt_flat == 1)).item()

    intersection = tp
    union = tp + fp + fn
    total_pixels = pred_flat.numel()
    return tp, tn, fp, fn, intersection, union, total_pixels

In [65]:
# 1. Initialize running accumulators (Takes virtually 0 RAM)
total_tp = 0
total_tn = 0
total_fp = 0
total_fn = 0
total_batches = 0

model.eval()
with torch.no_grad():
    for batch_imgs, batch_masks in tqdm(eval_loader, total=len(eval_loader)):
        # Move input batch to GPU asynchronously
        batch_imgs = batch_imgs.to(device, non_blocking=True)

        # Forward pass on GPU
        outputs = model(batch_imgs)

        # FIX: Handle HuggingFace SegFormer output object vs Standard PyTorch Tensor outputs
        logits = outputs.logits if hasattr(outputs, 'logits') else outputs

        # 2. FIX: If logits are downsampled (like SegFormer), upsample them to match batch_masks
        if logits.shape[-2:] != batch_masks.shape[-2:]:
            logits = F.interpolate(
                logits, 
                size=batch_masks.shape[-2:], 
                mode='bilinear', 
                align_corners=False
            )

        # Threshold directly on the GPU (using logits now)
        preds = (logits > threshold).int()
        batch_masks = batch_masks.int()

        # 2. CALCULATE RUNNING METRICS ON THE FLY
        tp, tn, fp, fn, intersection, union, total_px = get_batch_stats(preds, batch_masks.to(device))
        total_tp += tp
        total_tn += tn
        total_fp += fp
        total_fn += fn
        
        total_batches += 1

# ==========================================
# 3. FINAL DATASET-WIDE REDUCTION
# ==========================================
iou = total_tp / (total_tp + total_fp + total_fn + 1e-6)
dice_score = (2.0 * total_tp) / (2.0 * total_tp + total_fp + total_fn + 1e-6)
precision = total_tp / (total_tp + total_fp + 1e-6)
recall = total_tp / (total_tp + total_fn + 1e-6)
f1_score = 2.0 * precision * recall / (precision + recall + 1e-6)

print(f"IoU: {iou:.4f} | Dice: {dice_score:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1_score:.4f}")

100%|██████████| 278/278 [01:17<00:00,  3.57it/s]

IoU: 0.6313 | Dice: 0.7739 | Precision: 0.7938 | Recall: 0.7551 | F1: 0.7739
